# Data Splitting (Per-Sentence Distribution)
Notebook ini dikonfigurasi mengikuti pedoman `data_split_plan_v3.md`.

Tujuan utama:
1. **Setiap kalimat (gloss) mutlak harus ada** di set `train.csv`, `dev.csv`, dan `test.csv`.
2. **Zero Leakage**: Tidak boleh ada grup (`signer_sentence`) yang sama melintasi *Train, Dev, Test*.

In [16]:
import pandas as pd
import glob
import re

## 1. Load Excel & Mapping

In [17]:
excel_path = "Gloss dan Tanda Dataset.xlsx"
df_excel = pd.read_excel(excel_path)
display(df_excel.head())

# TODO: Sesuaikan nama kolom jika berbeda di file Excel sebenarnya
col_sentence = 'ID'      # Ubah sesuai header di Excel (misal: 'ID Tanda' atau 'Sentence ID')
col_gloss = 'Gloss'      # Ubah sesuai header di Excel
col_text = 'Kalimat'     # Tambahan untuk teks asli

mapping = {}
mapping_text = {}
if col_sentence in df_excel.columns and col_gloss in df_excel.columns:
    for _, row in df_excel.dropna(subset=[col_sentence]).iterrows():
        mapping[str(row[col_sentence]).strip()] = str(row[col_gloss]).strip()
        if col_text in df_excel.columns:
            mapping_text[str(row[col_sentence]).strip()] = str(row[col_text]).strip()
else:
    print("PERINGATAN: Kolom tidak ditemukan. Cek kembali variabel col_sentence dan col_gloss!")

,No.,ID,Kalimat,Gloss,N Gloss,Tanda-1,Tanda-2,Tanda-3,Tanda-4,Tanda-5,Tanda-6,Tanda-7,Tanda-8,Tanda-9,N Tanda
0,1.0,S001,Di mana ayah sama Ibu?,AYAH SAMA IBU MANA,4.0,AYAH,SAMA,IBU,MANA,NaN,NaN,NaN,NaN,NaN,4.0
1,2.0,S002,Dia anak baik sehingga banyak orang menyukainya,DIA ANAK BAIK SAMPAI BANYAK ORANG SUKA,7.0,DIA,ANAK,BAIK,SAMPAI,BANYAK,ORANG,SUKA,NaN,NaN,7.0
2,3.0,S003,Apa kamu pernah membaca novel bahasa inggris?,APA KAMU PERNAH BACA NOVEL B.INGGRIS,6.0,APA,KAMU,PERNAH,BACA,NOVEL,B. INGGRIS,NaN,NaN,NaN,6.0
3,4.0,S004,"Badan ku gemuk, tapi badan adik kurus",BADAN AKU GEMUK TAPI BADAN ADIK KURUS,7.0,BADAN,AKU,GEMUK,TAPI,BADAN,ADIK,KURUS,NaN,NaN,7.0
4,5.0,S005,"Saya sering pusing, saya harus periksa ke mana?","AKU PUSING SERING, AKU HARUS PERIKSA MANA",7.0,AKU,PUSING,SERING,AKU,HARUS,PERIKSA,MANA,NaN,NaN,7.0


## 2. Parse ID dari Dataset

In [18]:
data_dir = "../data"
file_paths = glob.glob(f"{data_dir}/**/*.*", recursive=True)

id_pattern = re.compile(r'(P\d{2}_S\d{3}_R\d{2})')

all_ids = []
for p in file_paths:
    match = id_pattern.search(p)
    if match:
        all_ids.append(match.group(1))

all_ids = sorted(list(set(all_ids)))
print(f"Total unique IDs found: {len(all_ids)}")

if not all_ids:
    print("Menggunakan data dummy...")
    # Generate dummy data: 5 signer, masing-masing melakukan 10 kalimat berbeda sebanyak 2x repetisi
    all_ids = [
        f"P{signer:02d}_S{sentence:03d}_R{rep:02d}"
        for signer in range(1, 6)      
        for sentence in range(1, 11)   
        for rep in range(1, 3)         
    ]

Total unique IDs found: 750


## 3. Build DataFrame

In [19]:
data = []
for file_id in all_ids:
    parts = file_id.split('_')
    if len(parts) == 3:
        signer = parts[0]
        sentence = parts[1]
        repetition = parts[2]
        
        # Group = signer + sentence 
        group = f"{signer}_{sentence}"
        
        # Ambil gloss dari mapping
        gloss = mapping.get(sentence, f"UNKNOWN_{sentence}")
        text = mapping_text.get(sentence, f"UNKNOWN_{sentence}")
        
        data.append({
            'id': file_id,
            'gloss': gloss,
            'text': text,
            'signer': signer,
            'sentence': sentence,
            'group': group
        })

df = pd.DataFrame(data)
display(df.head())

,id,gloss,signer,sentence,group
0,P01_S001_R01,AYAH SAMA IBU MANA,P01,S001,P01_S001
1,P01_S001_R02,AYAH SAMA IBU MANA,P01,S001,P01_S001
2,P01_S001_R03,AYAH SAMA IBU MANA,P01,S001,P01_S001
3,P01_S001_R04,AYAH SAMA IBU MANA,P01,S001,P01_S001
4,P01_S001_R05,AYAH SAMA IBU MANA,P01,S001,P01_S001


## 4. Per-Sentence Split Algorithm
Memutar urutan alokasi `dev` dan `test` agar `signer` tertentu tidak menumpuk di subset evaluasi (menggunakan metode *Least-Used Assignment*).

In [20]:
train_ids = []
dev_ids = []
test_ids = []

unique_sentences = sorted(df['sentence'].unique())

# Tracker untuk melihat berapa kali seorang signer masuk ke Dev dan Test
dev_signer_counts = {signer: 0 for signer in df['signer'].unique()}
test_signer_counts = {signer: 0 for signer in df['signer'].unique()}

for sent in unique_sentences:
    df_sent = df[df['sentence'] == sent]
    unique_signers = sorted(df_sent['signer'].unique())
    
    if len(unique_signers) < 3:
        print(f"PERINGATAN: Kalimat {sent} hanya dilakukan oleh {len(unique_signers)} signer. Tidak ideal untuk displit 3 arah.")
    
    # Pilih dev_candidate dengan prioritas yang count-nya paling kecil (dan lexicographical terkecil jika seri)
    dev_candidate = sorted(unique_signers, key=lambda s: (dev_signer_counts[s], s))[0]
    dev_signer_counts[dev_candidate] += 1
    
    # Pilih test_candidate dari sisa signer dengan count terkecil
    test_candidates = [s for s in unique_signers if s != dev_candidate]
    if test_candidates:
        test_candidate = sorted(test_candidates, key=lambda s: (test_signer_counts[s], s))[0]
        test_signer_counts[test_candidate] += 1
    else:
        test_candidate = None
        
    train_candidates = [s for s in unique_signers if s not in (dev_candidate, test_candidate)]
    
    # Alokasi semua ID repetisi ke masing-masing himpunan
    dev_ids.extend(df_sent[df_sent['signer'] == dev_candidate]['id'].tolist())
    if test_candidate:
        test_ids.extend(df_sent[df_sent['signer'] == test_candidate]['id'].tolist())
    for tc in train_candidates:
        train_ids.extend(df_sent[df_sent['signer'] == tc]['id'].tolist())

# Terapkan ID ke final DataFrame
df_train = df[df['id'].isin(train_ids)].copy()
df_dev = df[df['id'].isin(dev_ids)].copy()
df_test = df[df['id'].isin(test_ids)].copy()

print(f"Total keseluruhan data: {len(df)}")
print(f"Train size: {len(df_train)} ({len(df_train)/len(df):.1%})")
print(f"Dev size  : {len(df_dev)} ({len(df_dev)/len(df):.1%})")
print(f"Test size : {len(df_test)} ({len(df_test)/len(df):.1%})")


Total keseluruhan data: 750
Train size: 450 (60.0%)
Dev size  : 150 (20.0%)
Test size : 150 (20.0%)


## 5. Validasi Ketat (Wajib Lulus)

In [21]:
# 1. Coverage Test
train_glosses = set(df_train['gloss'])
dev_glosses = set(df_dev['gloss'])
test_glosses = set(df_test['gloss'])
all_glosses = set(df['gloss'])

print("--- Validasi Coverage (100% Kelas harus ada) ---")
assert train_glosses == all_glosses, f"Train missing glosses! Miss: {all_glosses - train_glosses}"
assert dev_glosses == all_glosses, f"Dev missing glosses! Miss: {all_glosses - dev_glosses}"
assert test_glosses == all_glosses, f"Test missing glosses! Miss: {all_glosses - test_glosses}"
print("OK! Kelas Gloss di Train, Dev, dan Test tercover 100%.")

# 2. Leakage Test
train_groups = set(df_train['group'])
dev_groups = set(df_dev['group'])
test_groups = set(df_test['group'])

print("\n--- Validasi Leakage (Tidak ada grup partisipan yang tumpang tindih) ---")
assert train_groups.isdisjoint(dev_groups), f"Leakage Train-Dev: {train_groups.intersection(dev_groups)}"
assert train_groups.isdisjoint(test_groups), f"Leakage Train-Test: {train_groups.intersection(test_groups)}"
assert dev_groups.isdisjoint(test_groups), f"Leakage Dev-Test: {dev_groups.intersection(test_groups)}"
print("OK! Zero Data Leakage. Tidak ada partisipan yang bocor antar set evaluasi.")

--- Validasi Coverage (100% Kelas harus ada) ---
OK! Kelas Gloss di Train, Dev, dan Test tercover 100%.

--- Validasi Leakage (Tidak ada grup partisipan yang tumpang tindih) ---
OK! Zero Data Leakage. Tidak ada partisipan yang bocor antar set evaluasi.


## 6. Output CSV

In [22]:
output_columns = ['id', 'gloss']
output_columns_txt = ['id', 'gloss', 'text']

df_train[output_columns].to_csv('train.csv', index=False, sep='|')
df_dev[output_columns].to_csv('dev.csv', index=False, sep='|')
df_test[output_columns].to_csv('test.csv', index=False, sep='|')

output_columns_txt = ['id', 'gloss', 'text']
df_train[output_columns_txt].to_csv('sd_train_list.txt', index=False, sep='|')
df_dev[output_columns_txt].to_csv('sd_dev_list.txt', index=False, sep='|')
df_test[output_columns_txt].to_csv('sd_test_list.txt', index=False, sep='|')

print("Selesai! File train.csv, dev.csv, test.csv, serta file list TXT siap digunakan.")

Selesai! File train.csv, dev.csv, dan test.csv siap digunakan.
